In [1]:
!pip install pydicom opencv-python tqdm

In [9]:
import pandas as pd

csv_path = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv"
df = pd.read_csv(csv_path)

print("CSV Loaded Successfully")
print("Total rows:", len(df))
print(df.head())

CSV Loaded Successfully
Total rows: 67914
                           image_id          class_name  class_id rad_id  \
0  50a418190bc3fb1ef1633bf9678929b3          No finding        14    R11   
1  21a10246a5ec7af151081d0cd6d65dc9          No finding        14     R7   
2  9a5094b2563a1ef3ff50dc5c7ff71345        Cardiomegaly         3    R10   
3  051132a778e61a86eb147c7c6f564dfe  Aortic enlargement         0    R10   
4  063319de25ce7edb9b1c6b8881290140          No finding        14    R10   

    x_min   y_min   x_max   y_max  
0     NaN     NaN     NaN     NaN  
1     NaN     NaN     NaN     NaN  
2   691.0  1375.0  1653.0  1831.0  
3  1264.0   743.0  1611.0  1019.0  
4     NaN     NaN     NaN     NaN  


In [10]:
import numpy as np

# Separate abnormal and normal
abnormal_df = df[df['class_id'] != 14]   # 14 = No finding in VinBigData
normal_df = df[df['class_id'] == 14]

abnormal_ids = abnormal_df['image_id'].unique()
normal_ids = normal_df['image_id'].unique()

print("Total abnormal images:", len(abnormal_ids))
print("Total normal images:", len(normal_ids))

# Select 3000 abnormal
selected_abnormal = np.random.choice(abnormal_ids, 3000, replace=False)

# Select 2000 normal
selected_normal = np.random.choice(normal_ids, 2000, replace=False)

selected_ids = np.concatenate([selected_abnormal, selected_normal])

selected_df = df[df['image_id'].isin(selected_ids)]

selected_df.to_csv("train_selected.csv", index=False)

print("Total selected images:", len(selected_ids))

Total abnormal images: 4394
Total normal images: 10606
Total selected images: 5000


In [11]:
len(selected_ids)

5000

In [13]:
import os

train_folder = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train"
print(os.listdir(train_folder)[:5])

['4d390e07733ba06e5ff07412f09c0a92.dicom', '289f69f6462af4933308c275d07060f0.dicom', '68335ee73e67706aa59b8b55b54b11a4.dicom', '7ecd6f67f649f26c05805c8359f9e528.dicom', '2229148faa205e881cf0d932755c9e40.dicom']


In [15]:
import os
import pydicom
import cv2
from tqdm import tqdm

# Create output folder
os.makedirs("images", exist_ok=True)

dicom_folder = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train"

for img_id in tqdm(selected_ids):
    
    dicom_path = os.path.join(dicom_folder, img_id + ".dicom")
    
    ds = pydicom.dcmread(dicom_path)
    img = ds.pixel_array
    
    # Normalize (VERY IMPORTANT)
    img = (img - img.min()) / (img.max() - img.min())
    img = (img * 255).astype("uint8")
    
    # Resize to 512x512
    img = cv2.resize(img, (512, 512))
    
    cv2.imwrite(f"images/{img_id}.png", img)

print("✅ 5000 Images Converted Successfully")

100%|██████████| 5000/5000 [1:56:31<00:00,  1.40s/it]  

✅ 5000 Images Converted Successfully


In [16]:
len(os.listdir("images"))

5000

In [17]:
import os
import pandas as pd
import cv2
import pydicom
from tqdm import tqdm

# Load filtered CSV
df = pd.read_csv("train_selected.csv")

os.makedirs("labels", exist_ok=True)

dicom_folder = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train"

for img_id in tqdm(df['image_id'].unique()):
    
    img_df = df[df['image_id'] == img_id]
    
    # Read original DICOM to get original size
    dicom_path = os.path.join(dicom_folder, img_id + ".dicom")
    ds = pydicom.dcmread(dicom_path)
    orig_h, orig_w = ds.pixel_array.shape
    
    label_path = f"labels/{img_id}.txt"
    
    with open(label_path, "w") as f:
        
        for _, row in img_df.iterrows():
            
            # Skip normal class (class_id = 14)
            if row['class_id'] == 14:
                continue
            
            x_min = row['x_min']
            y_min = row['y_min']
            x_max = row['x_max']
            y_max = row['y_max']
            
            # Scale to 512x512
            x_min = x_min * (512 / orig_w)
            x_max = x_max * (512 / orig_w)
            y_min = y_min * (512 / orig_h)
            y_max = y_max * (512 / orig_h)
            
            # Convert to YOLO format
            x_center = ((x_min + x_max) / 2) / 512
            y_center = ((y_min + y_max) / 2) / 512
            width = (x_max - x_min) / 512
            height = (y_max - y_min) / 512
            
            class_id = int(row['class_id'])
            
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("✅ YOLO Labels Created Successfully")

100%|██████████| 5000/5000 [1:49:20<00:00,  1.31s/it]  

✅ YOLO Labels Created Successfully


In [18]:
len(os.listdir("labels"))

5000

In [19]:
with open("labels/" + df['image_id'].iloc[0] + ".txt") as f:
    print(f.read())

In [20]:
class_names = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis"
]

with open("dataset.yaml", "w") as f:
    f.write("train: images\n")
    f.write("val: images\n\n")
    f.write("nc: 14\n")
    f.write("names:\n")
    for name in class_names:
        f.write(f"  - {name}\n")

print("✅ dataset.yaml created successfully")

✅ dataset.yaml created successfully


In [21]:
with open("dataset.yaml", "r") as f:
    print(f.read())

train: images
val: images

nc: 14
names:
  - Aortic enlargement
  - Atelectasis
  - Calcification
  - Cardiomegaly
  - Consolidation
  - ILD
  - Infiltration
  - Lung Opacity
  - Nodule/Mass
  - Other lesion
  - Pleural effusion
  - Pleural thickening
  - Pneumothorax
  - Pulmonary fibrosis



In [23]:
import os
import shutil

# Create clean export folder
os.makedirs("Final_Module1_Output", exist_ok=True)

# Copy only necessary folders/files
shutil.copytree("images", "Final_Module1_Output/images")
shutil.copytree("labels", "Final_Module1_Output/labels")
shutil.copy("dataset.yaml", "Final_Module1_Output/dataset.yaml")

# Optional
if os.path.exists("train_selected.csv"):
    shutil.copy("train_selected.csv", "Final_Module1_Output/train_selected.csv")

# Now zip only this folder
shutil.make_archive("VinDr_Module1_Output", 'zip', "Final_Module1_Output")

print("✅ Clean ZIP file created successfully")

✅ Clean ZIP file created successfully
